In [1]:
class MyStack:
    def pop(self):
        return "Removed"

def remove_last(obj):
    return obj.pop()

print(remove_last([1, 2, 3]))
print(remove_last(MyStack()))

3
Removed


In [2]:
class MySequence:
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        return self.data[index]

seq = MySequence([10, 20, 30])
print(len(seq))
print(seq[1])
for item in seq:
    print(item)

3
20
10
20
30


In [3]:
class Foo:
    def __getitem__(self, pos):
        return range(0, 30, 10)[pos]

f = Foo()
f[1]

10

In [4]:
for i in f:
    print(i)

0
10
20


In [5]:
20 in f

True

In [6]:
15 in f

False

In [9]:
PyObject *
PyObject_GetIter (PyObject *o)
{
    PyTypeObject *t = Py_TYPE(o);
    getiterfunc f;
    f = t->tp_iter;
    if (f == NULL) {
        if (PySequence_Check(o))
            return PySeqIter_New(o);
        return type_error("'%.200s' object is not iterable", o);
    }
    else {
        PyObject *res = (*f)(o);
        if (res != NULL && ! PyIter_Check(res)) {
            PyErr_Format(PyExc_TypeError,
                "%.200s.__iter__() must return an iterator, not %.200s",
                o->ob_type->tp_name, res->ob_type->tp_name);
            Py_SETREF (res, NULL);
        }
        return res;
    }
}

SyntaxError: invalid syntax (2371662689.py, line 1)

In [11]:
import collections
Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

In [14]:
from random import shuffle
from frenchdeck import FrenchDeck
deck = FrenchDeck()
shuffle(deck)

ModuleNotFoundError: No module named 'frenchdeck'

In [19]:
def set_card(deck, position, card):
    deck._cards[position] = card

FrenchDeck.__setitem__ = set_card
deck = FrenchDeck()
shuffle(deck)
deck[:5]

[Card(rank='A', suit='spades'),
 Card(rank='7', suit='diamonds'),
 Card(rank='10', suit='hearts'),
 Card(rank='8', suit='spades'),
 Card(rank='3', suit='spades')]

In [25]:
import collections
from collections.abc import MutableSequence

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck2(MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

    def __setitem__(self, position, value):
        self._cards[position] = value

    def __delitem__(self, position):
        del self._cards[position]

    def insert(self, position, value):
        self._cards.insert(position, value)

deck2 = FrenchDeck2()
print(len(deck2))
print(deck2[0])

52
Card(rank='2', suit='spades')


In [26]:
from abc import ABC, abstractmethod

class Drawable(ABC):
    @abstractmethod
    def draw(self):
        pass

class ExternalPainter:
    def draw(self):
        print("Drawing")

Drawable.register(ExternalPainter)
obj = ExternalPainter()
print(isinstance(obj, Drawable))

True


In [27]:
from collections.abc import Sized

class MyCollection:
    def __len__(self):
        return 10

Sized.register(MyCollection)
obj = MyCollection()
print(isinstance(obj, Sized))

True


In [28]:
import abc

class Tombola(abc.ABC):

    @abc.abstractmethod
    def load(self, iterable):
        """Add items from an iterable."""

    @abc.abstractmethod
    def pick(self):
        """Remove item at random, returning it.
        This method should raise `LookupError` when the instance is empty.
        """

    def loaded(self):
        """Return `True` if there's at least 1 item, `False` otherwise."""
        return bool(self.inspect())

    def inspect(self):
        """Return a sorted tuple with the items currently inside."""
        items = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(sorted(items))

In [36]:
import random

# 이전 셀(1NtAUwDnJsit)에 Tombola 클래스가 이미 정의되어 있으므로 import 문이 필요 없습니다.
class BingoCage(Tombola):

    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

    def __call__(self):
        self.pick()

In [38]:
import random

class LotteryBlower(Tombola):

    def __init__(self, iterable):
        self._balls = list(iterable)

    def load(self, iterable):
        self._balls.extend(iterable)

    def pick(self):
        try:
            position = random.randrange(len(self._balls))
        except ValueError:
            raise LookupError('pick from empty BingoCage')
        return self._balls.pop(position)

    def loaded(self):
        return bool(self._balls)

    def inspect(self):
        return tuple(sorted(self._balls))

In [40]:
from random import randrange

@Tombola.register
class TomboList(list):

    def pick(self):
        if self:
            position = randrange(len(self))
            return self.pop(position)
        else:
            raise LookupError('pop from empty TomboList')

    load = list.extend

    def loaded(self):
        return bool(self)

    def inspect(self):
        return tuple(sorted(self))

In [47]:
import doctest
from tombola import Tombola

import bingo, lotto, tombolist, drum

TEST_FILE = 'tombola_tests.rst'
TEST_MSG = '{0:16} {1.attempted:2} tests, {1.failed:2} failed {2}'

def main(argv):
    verbose = '-v' in argv
    real_subclasses = Tombola.__subclasses__()
    virtual_subclasses = list(Tombola._abc_registry)

    for cls in real_subclasses + virtual_subclasses:
        test(cls, verbose)

def test(cls, verbose=False):
    res = doctest.testfile(
        TEST_FILE,
        globs={'Concrete Tombola': cls},
        verbose=verbose,
        optionflags=doctest.REPORT_ONLY_FIRST_FAILURE)
    tag = 'FAIL' if res.failed else 'OK'
    print(TEST_MSG.format(cls.__name__, res, tag))

if __name__ == '__main__':
    import sys
    main(sys.argv)

테스트 파일 'tombola_tests.rst'이 존재하지 않아 BingoCage 클래스를 테스트할 수 없습니다.
테스트 파일 'tombola_tests.rst'이 존재하지 않아 LotteryBlower 클래스를 테스트할 수 없습니다.
